In [ ]:
# install libraries

%pip install pandas
%pip install geopandas pyogrio

In [ ]:
import geopandas as gpd

# open the greater-london map data with geographical pandas
f = "../data/raw/greater-london-street-map.gpkg"
pts = gpd.read_file(f, layer="gis_osm_pois_free")
pts_table = pts.drop(columns="geometry")  
pts_table


,osm_id,code,fclass,name
0,99884,2906,waste_basket,
1,108539,2566,bicycle_rental,Windsor Terrace
2,388826,2701,tourist_info,
3,451152,2304,pub,King of Prussia
4,451153,2301,restaurant,Central Restaurant
...,...,...,...,...
106188,14409939515,2120,doctors,Dr Ozan Demir - Consultant Cardiologist
106189,14410060832,2033,recycling_clothes,
106190,14410127556,2742,viewpoint,
106191,14410477201,2301,restaurant,Ozen Firin


Below is the approach which was taught in project labs (using standard pandas rather than geographical)

In [10]:
# turn the map file into a .csv file
import geopandas as gpd

places = gpd.read_file("../data/raw/greater-london-street-map.gpkg", layer="gis_osm_pois_free")
places["lat"] = places.geometry.y
places["lon"] = places.geometry.x
places.drop(columns="geometry").to_csv("../data/raw/greater-london-street-map.csv", index=False)

In [ ]:
# open the csv file and call it places
import pandas as pd
places = pd.read_csv("../data/raw/greater-london-street-map.csv")
places.head()


,osm_id,code,fclass,name,lat,lon
0,99884,2906,waste_basket,NaN,51.524364,-0.152816
1,108539,2566,bicycle_rental,Windsor Terrace,51.529125,-0.093388
2,388826,2701,tourist_info,NaN,51.530421,-0.155825
3,451152,2304,pub,King of Prussia,51.600840,-0.194608
4,451153,2301,restaurant,Central Restaurant,51.602031,-0.193503


In [17]:
# number of rows and columns
places.shape

(106193, 6)

In [18]:
# what kind of data is in each column and how many cells are filled?
places.info()

<class 'pandas.DataFrame'>
RangeIndex: 106193 entries, 0 to 106192
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   osm_id  106193 non-null  int64  
 1   code    106193 non-null  int64  
 2   fclass  106193 non-null  str    
 3   name    48429 non-null   str    
 4   lat     106193 non-null  float64
 5   lon     106193 non-null  float64
dtypes: float64(2), int64(2), str(2)
memory usage: 4.9 MB


In [20]:
# list every kind of place in the file and how many of each
places["fclass"].value_counts()

fclass
bench            21803
waste_basket     10037
post_box          8042
restaurant        6215
cafe              4898
                 ...  
caravan_site         1
lighthouse           1
golf_course          1
wayside_cross        1
camp_site            1
Name: count, Length: 132, dtype: int64

In [ ]:
# keep only the places where people would want to meet
keep = ["cafe", "restaurant", "pub", "bar", "fast_food", "park", "library", "community_centre"]
places = places[places["fclass"].isin(keep)]

places.shape

(18159, 6)

We originally started with 106,193 different records, but now we're left with 18,159 after keeping only the places where we think people would want to meet. That means we removed 88,034 records. 

In [24]:
# list the number of blank cells in each column
places.isna().sum()

osm_id      0
code        0
fclass      0
name      406
lat         0
lon         0
dtype: int64

The only blank cells are in the name column, and there are 406 of them. 

In [25]:
# which kind of places have no names?
places[places["name"].isna()]["fclass"].value_counts()

fclass
cafe                162
restaurant          107
fast_food            83
bar                  23
community_centre     15
pub                  12
library               4
Name: count, dtype: int64

The blank named places are quite spread out, so we can drop these.

In [ ]:
# drop places with no name
places = places.dropna(subset=["name"])

# check so that there are no remaining empty cells
places.isna().sum()

osm_id    0
code      0
fclass    0
name      0
lat       0
lon       0
dtype: int64

In [ ]:
# check for any duplicated records, but as they all have a different osm_id then only check the non_unique fields

# remove stray spaces at either end of each name
places["name"] = places["name"].str.strip()

# check for duplicate records
places.duplicated(subset=["name", "fclass", "lat", "lon"]).sum()

np.int64(2)

We have 2 duplicate fields so remove these

In [ ]:
# keep one copy of a duplicate record and drop the rest
places = places.drop_duplicates(subset=["name", "fclass", "lat", "lon"])

places.duplicated(subset=["name", "fclass", "lat", "lon"]).sum()

np.int64(0)

In [35]:
# Every kind of place and how often it appears
places["fclass"].value_counts()

fclass
restaurant          6107
cafe                4735
fast_food           4271
pub                 1276
bar                  896
community_centre     293
library              164
park                   9
Name: count, dtype: int64

We have all of the correct classes, but the park is very low because we're using normal pandas instead of GeoPandas, which draws an outline instead of a single coordinate point. So we'll have to look at this. 

Lets check impossible values now.

In [34]:
# Smallest and largest latitude and longitude
places[["lat", "lon"]].describe()

,lat,lon
count,17751.000000,17751.000000
mean,51.505092,-0.127447
std,0.059581,0.111139
min,51.300346,-0.489050
25%,51.477519,-0.182345
50%,51.511123,-0.122556
75%,51.535806,-0.070626
max,51.681780,0.251602


After some research, London roughly has latitude from 51.28 to 51.7 and longitude from -0.52 to 0.34. Our minimum and maximum values are within that range, so we don't have to remove any. 

In [36]:
# save the clean table

places.to_csv("../data/processed/london_places.csv", index=False)
print("Saved", len(places), "rows to london_places.csv")

Saved 17751 rows to london_places.csv
